In [1]:
# =============================================================================
# CELL 1 — Imports
# =============================================================================

from __future__ import annotations

import importlib
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

import src.pago_pipeline.ncbi_snapshot as ncbi_snapshot_module

# Reload the pipeline module so notebook reruns pick up local code changes.
ncbi_snapshot_module = importlib.reload(ncbi_snapshot_module)

SnapshotMode = ncbi_snapshot_module.SnapshotMode
get_snapshot_xml_file_path = ncbi_snapshot_module.get_snapshot_xml_file_path
resolve_ncbi_protein_xml_snapshot = (
    ncbi_snapshot_module.resolve_ncbi_protein_xml_snapshot
)

from src.pago_pipeline.storage import sha256_of_file, sha256_of_lines

In [2]:
# =============================================================================
# CELL 2 — Load environment and resolve project root
# =============================================================================
dotenv_path = find_dotenv(usecwd=False)

if not dotenv_path:
    raise FileNotFoundError(
        "Could not find a .env file while walking up parent directories. "
        "Place .env with your NCBI email and optional API key at the project root."
    )

load_dotenv(dotenv_path=dotenv_path, override=True)

PROJECT_ROOT = Path(dotenv_path).resolve().parent
NCBI_EMAIL = os.getenv("NCBI_EMAIL")
NCBI_API_KEY = os.getenv("NCBI_API_KEY")

if not NCBI_EMAIL:
    raise ValueError(
        "NCBI_EMAIL was not found in the environment. "
        "Please define it in your .env file."
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"NCBI email configured: {bool(NCBI_EMAIL)}")
print(f"NCBI API key configured: {bool(NCBI_API_KEY)}")

Project root: C:\Programming\Python\pAgo-project
NCBI email configured: True
NCBI API key configured: True


In [3]:
# =============================================================================
# CELL 3 — Define XML snapshot configuration
# =============================================================================

# Notebook 01 expects notebook 00 to have already materialized the upstream
# protein UID snapshot in the source snapshot root directory below.
SOURCE_UID_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "01-raw" / "protein_uid_snapshots"
)
XML_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "01-raw" / "protein_xml_snapshots"
)

XML_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create
MAX_RETRY_ATTEMPTS = 5

XML_BATCH_SIZE = 100
XML_REQUEST_DELAY_SECONDS = None

UPDATE_LATEST_DIRECTORY = True

print(f"Source UID snapshot root directory: {SOURCE_UID_SNAPSHOT_ROOT_DIRECTORY}")
print(f"XML snapshot root directory: {XML_SNAPSHOT_ROOT_DIRECTORY}")
print(f"XML snapshot mode: {XML_SNAPSHOT_MODE}")
print("Expected upstream UID snapshot: latest snapshot produced by notebook 00")

Source UID snapshot root directory: C:\Programming\Python\pAgo-project\data\01-raw\protein_uid_snapshots
XML snapshot root directory: C:\Programming\Python\pAgo-project\data\01-raw\protein_xml_snapshots
XML snapshot mode: reuse_latest_or_create
Expected upstream UID snapshot: latest snapshot produced by notebook 00


In [4]:
# =============================================================================
# CELL 4 — Resolve active XML snapshot
# =============================================================================

try:
    xml_snapshot_payload = resolve_ncbi_protein_xml_snapshot(
        snapshot_mode=XML_SNAPSHOT_MODE,
        snapshot_root_directory=XML_SNAPSHOT_ROOT_DIRECTORY,
        source_uid_snapshot_root_directory=SOURCE_UID_SNAPSHOT_ROOT_DIRECTORY,
        xml_batch_size=XML_BATCH_SIZE,
        max_retry_attempts=MAX_RETRY_ATTEMPTS,
        xml_request_delay_seconds=XML_REQUEST_DELAY_SECONDS,
        ncbi_email=NCBI_EMAIL,
        ncbi_api_key=NCBI_API_KEY,
        update_latest_directory=UPDATE_LATEST_DIRECTORY,
    )
except FileNotFoundError as exc:
    raise FileNotFoundError(
        "No upstream protein UID snapshot was found. "
        "Run notebook 00_ncbi_protein_ids_query.ipynb first, "
        "then rerun notebook 01_ncbi_protein_xml_snapshot.ipynb."
    ) from exc

xml_snapshot_directory = xml_snapshot_payload["snapshot_directory"]
manifest_file_path = xml_snapshot_payload["manifest_file_path"]
protein_uids_file_path = xml_snapshot_payload["protein_uids_file_path"]
xml_snapshot_manifest = xml_snapshot_payload["manifest"]
protein_uids = xml_snapshot_payload["protein_uids"]

xml_file_path = get_snapshot_xml_file_path(
    snapshot_directory=xml_snapshot_directory,
)
uid_dataset_sha256 = sha256_of_lines(
    text_lines=protein_uids,
    deduplicate_lines_preserving_order=False,
    sort_lines=False,
)

print(f"Resolved XML snapshot directory: {xml_snapshot_directory}")
print(f"Resolved XML UID count: {len(protein_uids)}")
print(f"Resolved XML batch count: {xml_snapshot_manifest['batch_count']}")

Latest XML snapshot is available. Reusing frozen snapshot.
Resolved XML snapshot directory: C:\Programming\Python\pAgo-project\data\01-raw\protein_xml_snapshots\latest
Resolved XML UID count: 41345
Resolved XML batch count: 414


In [5]:
# =============================================================================
# CELL 5 — Print XML snapshot summary
# =============================================================================

xml_file_sha256 = sha256_of_file(input_file_path=xml_file_path)

print("XML snapshot resolved successfully.")
print(f"Immutable XML snapshot directory: {xml_snapshot_directory}")
print(f"Consolidated XML file: {xml_file_path}")
print(f"Consolidated XML SHA-256: {xml_file_sha256}")
print(f"Total protein UIDs covered: {len(protein_uids)}")
print(f"Total XML batches used for consolidation: {xml_snapshot_manifest['batch_count']}")

XML snapshot resolved successfully.
Immutable XML snapshot directory: C:\Programming\Python\pAgo-project\data\01-raw\protein_xml_snapshots\latest
Consolidated XML file: C:\Programming\Python\pAgo-project\data\01-raw\protein_xml_snapshots\latest\protein_records.xml
Consolidated XML SHA-256: ada0563932d20f54f68c614df3c40d4fa3520038ad438a41cc471a1eacb970bf
Total protein UIDs covered: 41345
Total XML batches used for consolidation: 414


In [6]:
# =============================================================================
# CELL 6 — Print manifest-derived retrieval summary
# =============================================================================

print(f"Retrieved at UTC: {xml_snapshot_manifest['retrieved_at_utc']}")
print(f"Manifest batch size: {xml_snapshot_manifest['batch_size']}")
print(f"Manifest batch count: {xml_snapshot_manifest['batch_count']}")
print(
    "Manifest normalized protein UID count: "
    f"{xml_snapshot_manifest['normalized_protein_uid_count']}"
)
print(
    "Manifest consolidated record count: "
    f"{xml_snapshot_manifest['consolidated_record_count']}"
)
print(
    "Source UID snapshot relative path: "
    f"{xml_snapshot_manifest['source_uid_snapshot_relative_path']}"
)

Retrieved at UTC: 2026-04-09T00:51:02Z
Manifest batch size: 100
Manifest batch count: 414
Manifest normalized protein UID count: 41345
Manifest consolidated record count: 41345
Source UID snapshot relative path: snapshots\2026-04-06T20-42-12Z__q_891f443d754c


In [7]:
# =============================================================================
# CELL 7 — Print persisted file hashes
# =============================================================================

saved_manifest_sha256 = sha256_of_file(input_file_path=manifest_file_path)
saved_protein_uids_file_sha256 = sha256_of_file(
    input_file_path=protein_uids_file_path,
)

print("Persisted XML snapshot file hashes:")
print(f"Manifest SHA-256: {saved_manifest_sha256}")
print(f"Protein UIDs file SHA-256: {saved_protein_uids_file_sha256}")
print(f"Consolidated XML file SHA-256: {xml_file_sha256}")

Persisted XML snapshot file hashes:
Manifest SHA-256: 3bcd96d0ed3e51403b51558d14644f6b8a6c4e191ce34015f7f807f38143f229
Protein UIDs file SHA-256: c0ad9e0d797cc453887cddb14adea255592c18b5edc0319a39a9b012b96b1c09
Consolidated XML file SHA-256: ada0563932d20f54f68c614df3c40d4fa3520038ad438a41cc471a1eacb970bf


In [8]:
# =============================================================================
# CELL 8 — Print persisted manifest summary
# =============================================================================

print("Persisted XML snapshot metadata:")
print(f"Manifest path: {manifest_file_path}")
print(f"Protein UIDs file path: {protein_uids_file_path}")
print(f"Manifest XML file name: {xml_snapshot_manifest['xml_file_name']}")
print(f"Manifest XML SHA-256: {xml_snapshot_manifest['xml_file_sha256']}")
print(
    "Manifest immutable snapshot relative path: "
    f"{xml_snapshot_manifest['immutable_snapshot_relative_path']}"
)
print(
    "Manifest source UID snapshot relative path: "
    f"{xml_snapshot_manifest['source_uid_snapshot_relative_path']}"
)

Persisted XML snapshot metadata:
Manifest path: C:\Programming\Python\pAgo-project\data\01-raw\protein_xml_snapshots\latest\manifest.json
Protein UIDs file path: C:\Programming\Python\pAgo-project\data\01-raw\protein_xml_snapshots\latest\protein_uids.txt
Manifest XML file name: protein_records.xml
Manifest XML SHA-256: ada0563932d20f54f68c614df3c40d4fa3520038ad438a41cc471a1eacb970bf
Manifest immutable snapshot relative path: snapshots\2026-04-09T00-51-02Z__q_891f443d754c
Manifest source UID snapshot relative path: snapshots\2026-04-06T20-42-12Z__q_891f443d754c


In [9]:
# =============================================================================
# CELL 9 — Inspect first saved batch records
# =============================================================================

first_three_saved_batch_records = xml_snapshot_manifest["batches"][:3]

for saved_batch_record in first_three_saved_batch_records:
    print(f"Batch index: {saved_batch_record['batch_index']}")
    print(
        "Batch UID interval: "
        f"{saved_batch_record['batch_start_index']}"
        f"..{saved_batch_record['batch_end_index']}"
    )
    print(saved_batch_record["xml_payload_sha256"])
    print(saved_batch_record["protein_uid_count"])
    print("---")

Batch index: 1
Batch UID interval: 0..99
c50dc7d313f9a45befdacb9fa9840ace1579b80164b0c4a163a53c50522b0b3d
100
---
Batch index: 2
Batch UID interval: 100..199
c78bb11c8f9579eacffc8da1e22141e53c532c9d2bac2163432121ef9a159406
100
---
Batch index: 3
Batch UID interval: 200..299
f1c8005d85eab5e0d57e17375aec003faa90031fadffd5165abdce64268b1e6a
100
---


In [10]:
# =============================================================================
# CELL 10 — Validate persisted snapshot consistency
# =============================================================================

print("XML snapshot resolution completed successfully.")
print(f"Immutable XML snapshot directory: {xml_snapshot_directory}")
print(f"Consolidated XML file: {xml_file_path}")
print(f"Protein UIDs file: {protein_uids_file_path}")
print(f"Manifest file: {manifest_file_path}")
print(
    "Manifest XML file name matches saved path: "
    f"{xml_snapshot_manifest['xml_file_name'] == xml_file_path.name}"
)
print(
    "Manifest XML SHA-256 matches saved XML file: "
    f"{xml_snapshot_manifest['xml_file_sha256'] == xml_file_sha256}"
)
print(
    "Manifest UID SHA-256 matches resolved UID list: "
    f"{xml_snapshot_manifest['protein_uids_sha256'] == uid_dataset_sha256}"
)
print(
    "Manifest UID count matches resolved UID list: "
    f"{xml_snapshot_manifest['normalized_protein_uid_count'] == len(protein_uids)}"
)

XML snapshot resolution completed successfully.
Immutable XML snapshot directory: C:\Programming\Python\pAgo-project\data\01-raw\protein_xml_snapshots\latest
Consolidated XML file: C:\Programming\Python\pAgo-project\data\01-raw\protein_xml_snapshots\latest\protein_records.xml
Protein UIDs file: C:\Programming\Python\pAgo-project\data\01-raw\protein_xml_snapshots\latest\protein_uids.txt
Manifest file: C:\Programming\Python\pAgo-project\data\01-raw\protein_xml_snapshots\latest\manifest.json
Manifest XML file name matches saved path: True
Manifest XML SHA-256 matches saved XML file: True
Manifest UID SHA-256 matches resolved UID list: True
Manifest UID count matches resolved UID list: True


In [11]:
# =============================================================================
# CELL 11 — Print source UID snapshot provenance
# =============================================================================
print("Source UID snapshot provenance:")
print(
    "Source UID snapshot relative path: "
    f"{xml_snapshot_manifest['source_uid_snapshot_relative_path']}"
)
print(
    "Source UID snapshot directory name: "
    f"{xml_snapshot_manifest['source_uid_snapshot_directory_name']}"
)
print(
    "Source UID snapshot manifest SHA-256: "
    f"{xml_snapshot_manifest['source_uid_snapshot_manifest_sha256']}"
)
print(f"Source UID count: {xml_snapshot_manifest['source_uid_count']}")

Source UID snapshot provenance:
Source UID snapshot relative path: snapshots\2026-04-06T20-42-12Z__q_891f443d754c
Source UID snapshot directory name: 2026-04-06T20-42-12Z__q_891f443d754c
Source UID snapshot manifest SHA-256: d84a2f0183232e5eece4c24c07daffb95df265132e28a4f09a8f03f802b24ee2
Source UID count: 41345
